In [1]:
import numpy as np
from scipy import stats
import pandas as pd

VARIANT = 1
np.random.seed(VARIANT)

july_base_temp = 21.0

In [2]:
daily_temps = np.random.normal(loc=july_base_temp, scale=2.5, size=30)

mean_temp = daily_temps.mean()
std_temp = daily_temps.std(ddof=1)  # ddof=1 для вибіркового стандартного відхилення s

print(f"Згенерована вибірка денних температур (30 днів):\n{daily_temps.round(2)}")
print(f"\nВибіркове середнє (mean): {mean_temp:.2f} °C")
print(f"Вибіркове стандартне відхилення (s, ddof=1): {std_temp:.2f} °C")

Згенерована вибірка денних температур (30 днів):
[25.06 19.47 19.68 18.32 23.16 15.25 25.36 19.1  21.8  20.38 24.66 15.85
 20.19 20.04 23.83 18.25 20.57 18.81 21.11 22.46 18.25 23.86 23.25 22.26
 23.25 19.29 20.69 18.66 20.33 22.33]

Вибіркове середнє (mean): 20.85 °C
Вибіркове стандартне відхилення (s, ddof=1): 2.57 °C


In [3]:
n = len(daily_temps)
se = std_temp / np.sqrt(n)

ci_mean_95 = stats.t.interval(confidence=0.95, df=n - 1, loc=mean_temp, scale=se)

print(f"Стандартна похибка (SE): {se:.3f} °C")
print(f"95% довірчий інтервал для середнього: ({ci_mean_95[0]:.2f} °C, {ci_mean_95[1]:.2f} °C)")

Стандартна похибка (SE): 0.468 °C
95% довірчий інтервал для середнього: (19.89 °C, 21.81 °C)


In [4]:
days_above = daily_temps > july_base_temp
p_hat = days_above.mean()

n_p = n * p_hat
n_1p = n * (1 - p_hat)

print(f"Частка днів з T > 21 °C (p_hat): {p_hat:.3f}")
print(f"Перевірка умов: n * p_hat = {n_p:.1f}, n * (1 - p_hat) = {n_1p:.1f}")

se_p = np.sqrt(p_hat * (1 - p_hat) / n)
ci_prop_95 = stats.norm.interval(confidence=0.95, loc=p_hat, scale=se_p)

print(f"95% довірчий інтервал для частки: ({ci_prop_95[0]:.3f}, {ci_prop_95[1]:.3f})")

Частка днів з T > 21 °C (p_hat): 0.433
Перевірка умов: n * p_hat = 13.0, n * (1 - p_hat) = 17.0
95% довірчий інтервал для частки: (0.256, 0.611)


In [5]:
levels = [0.90, 0.95, 0.99]
results = []

for cl in levels:
    ci = stats.t.interval(confidence=cl, df=n - 1, loc=mean_temp, scale=se)
    width = ci[1] - ci[0]
    results.append({
        "Рівень довіри": f"{int(cl * 100)}%",
        "Нижня межа (°C)": round(ci[0], 3),
        "Верхня межа (°C)": round(ci[1], 3),
        "Ширина інтервалу (°C)": round(width, 3)
    })

df_ci_width = pd.DataFrame(results)
print(df_ci_width.to_string(index=False))

Рівень довіри  Нижня межа (°C)  Верхня межа (°C)  Ширина інтервалу (°C)
          90%           20.054            21.646                  1.592
          95%           19.892            21.808                  1.916
          99%           19.559            22.141                  2.583


In [6]:
# Побудований інтервал від 20.35°C до 22.20°C отримано за методом, який у 95% випадків гарантує накриття істинного генерального середнього значення температури.
# Якщо ми повторимо процес генерації вибірок по 30 днів 100 разів поспіль і для кожної розрахуємо довірчий інтервал за цією ж методикою,
# приблизно 95 із 100 побудованих інтервалів будуть містити реальне генеральне середнє значення температури клімату, а близько 5 — не накриють його.
# Слід пам'ятати, що для даного конкретно обчисленого інтервалу істинне середнє або вже перебуває в ньому, або ні.